# AI Fairness Evaluation Assignment

## Importing Libraries

In [3]:
# Importing necessary libraries
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

from skimage.metrics import structural_similarity as ssim
from skimage import exposure

Disclaimer:<br>
Steps 1-4A have been based on the following ChatGPT chat:<br>
[Source]()

## Step 1 - Pre-process Datasets

**Declaring paths to the datasets**

In [4]:
# Origingal dataset
dataset1Low = "data/dataset1/low"
dataset1High = "data/dataset1/high"
dataset2Input = "data/dataset2/input"
dataset2Target = "data/dataset2/target"

# Outputs
outputFolder = "outputs"
dataset1Processed = outputFolder + "/dataset1"
dataset2Processed = outputFolder + "/dataset2"
resized = outputFolder + "/resized"

**Getting image filenames**

In [ ]:
dataset1Files = os.listdir(dataset1Low)
dataset2Files = os.listdir(dataset2Input)

dataset1Files = [f for f in dataset1Files if f.endswith(".png")]
dataset2Files = [f for f in dataset2Files if f.endswith(".png")]

# Checking if the code works
print("Dataset 1 images:", len(dataset1Files))
print("Dataset 2 images:", len(dataset2Files))

Dataset 1 images: 15
Dataset 2 images: 90


**Enhancement code**

In [6]:
# CLAHE
def applyCLAHE(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(clipLimit = 2, tileGridSize = (8, 8))
    result = clahe.apply(gray)
    return cv2.cvtColor(result, cv2.COLOR_GRAY2BGR)

# Unsharp mask
def applyUnsharpMask(image):
    blurred = cv2.GaussianBlur(image, (0, 0), 1)
    result = cv2.addWeighted(image, 2.5, blurred, -1.5, 0)
    return result

# Box filter
def applyBoxFilter(image):
    return cv2.boxFilter(image, -1, (5, 5))

# Gaussian Smooth
def applyGaussianSmooth(image):
    return cv2.GaussianBlur(image, (5, 5), 1.5)

# Histogram Equalisation
def applyHistogramEqualisation(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    result = cv2.equalizeHist(gray)
    return cv2.cvtColor(result, cv2.COLOR_GRAY2BGR)

# Gamma correction
def applyGammaCorrection(image):
    gamma = 0.7
    imageNormalised = image / 255
    imageCorrected = np.power(imageNormalised, gamma)
    return np.uint8(imageCorrected * 255)

# Bilateral filter
def applyBilateralFilter(image):
    return cv2.bilateralFilter(image, 15, 75, 75)

# Denoise
def applyDenoise(image):
    return cv2.fastNlMeansDenoisingColored(image, None, 10, 10, 7, 21)

# Adaptive hist EQ
def applyAdaptiveHistEQ(image):
    imageRGB = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    result = exposure.equalize_adapthist(imageRGB / 255)
    result = np.uint8(result * 255)
    return cv2.cvtColor(result, cv2.COLOR_RGB2BGR)

# Saturation
def applySaturation(image):
    HSV = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    HSV[:, :, 1] = np.clip(HSV[:, :, 1] * 1.5, 0, 255)
    return cv2.cvtColor(HSV, cv2.COLOR_HSV2BGR)

# High pass filter
def applyHighPassFilter(image):
    blurred = cv2.GaussianBlur(image, (5, 5), 1.5)
    result = cv2.subtract(image, blurred)
    return result

**Choosing one method**

In [7]:
# This function can be changed if a different method of pre-processing is desired
def preprocessImage(image):
    result = applyCLAHE(image)
    return result

**Pre-processing datasets**

In [ ]:
# Dataset 1
for filename in dataset1Files:
    imagePath = dataset1Low + "/" + filename
    image = cv2.imread(imagePath)

    processed = preprocessImage(image)

    savePath = outputFolder + "/dataset1/" + filename
    cv2.imwrite(savePath, processed)

print("Dataset 1 pre-processing complete.")

# Dataset 2
for filename in dataset2Files:
    imagePath = dataset2Input + "/" + filename
    image = cv2.imread(imagePath)

    processed = preprocessImage(image)

    savePath = outputFolder + "/dataset2/" + filename
    cv2.imwrite(savePath, processed)

print("Dataset 2 pre-processing complete.")

Dataset 1 preprocessing complete.
Dataset 2 preprocessing complete.


## Step 2 - Metric Testing

**Metric functions**

In [13]:
# PSNR
def calculatePSNR(original, compared):
    MSE = np.mean((original - compared) ** 2)

    if MSE == 0:
        return 100

    maxPixel = 255
    PSNR = 20 * np.log10(maxPixel / np.sqrt(MSE))

    return PSNR

# SSIM
def calculateSSIM(original, compared):
    originalGray = cv2.cvtColor(original, cv2.COLOR_BGR2GRAY)
    comparedGray = cv2.cvtColor(compared, cv2.COLOR_BGR2GRAY)

    value = ssim(originalGray, comparedGray, data_range = 255)

    return value

# BRISQUE
def calculateBRISQUE(imagePath):
    try:
        from brisque import BRISQUE
        _BRISQUE = BRISQUE()
        score = _BRISQUE.get_score(imagePath)
        return score
    except:
        return np.nan

# NIQE
def calculateNIQE(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    mean = np.mean(gray)
    std = np.std(gray)

    score = abs(mean - 127.5) + abs(std - 64)

    return score

**Testing metrics**

In [ ]:
# Testing the metrics using the previously declared functions
filename = dataset1Files[0]

highImage = cv2.imread(dataset1High + "/" + filename)
lowImage = cv2.imread(dataset1Low + "/" + filename)

PSNRValue = calculatePSNR(highImage, lowImage)
SSIMValue = calculateSSIM(highImage, lowImage)
BRISQUEValue = calculateBRISQUE(dataset1Low + "/" + filename)
NIQEValue = calculateNIQE(lowImage)

print("PSNR:", PSNRValue)
print("SSIM:", SSIMValue)
print("BRISQUE:", BRISQUEValue)
print("NIQE:", NIQEValue)

PSNR: 27.789972533257817
SSIM: 0.24339526026144834
BRISQUE: nan
NIQE: 143.56711585615898


## Step 3 - Calculate Evaluation Metrics on Original Data

**Original data metrics**

In [17]:
# Dataset 1
resultsDataset1Original = []

for filename in dataset1Files:
    highPath = dataset1High + "/" + filename
    lowPath = dataset1Low + "/" + filename

    highImage = cv2.imread(highPath)
    lowImage = cv2.imread(lowPath)

    if highImage is None or lowImage is None:
        continue

    PSNRValue = calculatePSNR(highImage, lowImage)
    SSIMValue = calculateSSIM(highImage, lowImage)
    BRISQUEValue = calculateBRISQUE(lowPath)
    NIQEValue = calculateNIQE(lowImage)

    resultsDataset1Original.append(["dataset1", filename, "original", PSNRValue, SSIMValue, BRISQUEValue, NIQEValue])

dfDataset1Original = pd.DataFrame(resultsDataset1Original, columns = ["dataset", "filename", "type", "PSNR", "SSIM", "BRISQUE", "NIQE"])

display(dfDataset1Original.head())

# Dataset 2
resultsDataset2Original = []

for filename in dataset2Files:
    targetPath = dataset2Target + "/" + filename
    inputPath = dataset2Input + "/" + filename

    targetImage = cv2.imread(targetPath)
    inputImage = cv2.imread(inputPath)

    if targetImage is None or inputImage is None:
        continue

    PSNRValue = calculatePSNR(targetImage, inputImage)
    SSIMValue = calculateSSIM(targetImage, inputImage)
    BRISQUEValue = calculateBRISQUE(inputPath)
    NIQEValue = calculateNIQE(inputImage)

    resultsDataset2Original.append(["dataset2", filename, "original", PSNRValue, SSIMValue, BRISQUEValue, NIQEValue])

dfDataset2Original = pd.DataFrame(resultsDataset2Original, columns = ["dataset", "filename", "type", "PSNR", "SSIM", "BRISQUE", "NIQE"])

display(dfDataset2Original.head())

,dataset,filename,type,PSNR,SSIM,BRISQUE,NIQE
0,dataset1,1.png,original,27.789973,0.243395,NaN,143.567116
1,dataset1,111.png,original,27.977366,0.193150,NaN,161.444256
2,dataset1,146.png,original,28.539462,0.258655,NaN,154.398589
3,dataset1,179.png,original,27.797829,0.431618,NaN,164.216985
4,dataset1,22.png,original,27.655770,0.188468,NaN,159.177909


,dataset,filename,type,PSNR,SSIM,BRISQUE,NIQE
0,dataset2,uieb_10945.png,original,27.730635,0.775979,NaN,70.011382
1,dataset2,uieb_112_img_.png,original,28.463049,0.743476,NaN,54.578202
2,dataset2,uieb_117_img_.png,original,27.990357,0.740728,NaN,41.042570
3,dataset2,uieb_120_img_.png,original,27.832871,0.583641,NaN,42.685354
4,dataset2,uieb_12324.png,original,27.459798,0.473858,NaN,105.030567


**Original data summary**

In [18]:
originalResults = pd.concat([dfDataset1Original, dfDataset2Original])

originalSummary = originalResults.groupby(["dataset", "type"])[["PSNR", "SSIM", "BRISQUE", "NIQE"]].mean()

display(originalSummary)

,,PSNR,SSIM,BRISQUE,NIQE
dataset,type,,,,
dataset1,original,27.959167,0.189775,NaN,166.175521
dataset2,original,28.319352,0.818716,NaN,52.366277


**Pre-processed data metrics**

In [20]:
# Dataset 1
resultsDataset1Processed = []

for filename in dataset1Files:
    highPath = dataset1High + "/" + filename
    processedPath = outputFolder + "/dataset1/" + filename

    highImage = cv2.imread(highPath)
    processedImage = cv2.imread(processedPath)

    if highImage is None or processedImage is None:
        continue

    PSNRValue = calculatePSNR(highImage, processedImage)
    SSIMValue = calculateSSIM(highImage, processedImage)
    BRISQUEValue = calculateBRISQUE(processedPath)
    NIQEValue = calculateNIQE(processedImage)

    resultsDataset1Processed.append(["dataset1", filename, "processed", PSNRValue, SSIMValue, BRISQUEValue, NIQEValue])

dfDataset1Processed = pd.DataFrame(resultsDataset1Processed, columns = ["dataset", "filename", "type", "PSNR", "SSIM", "BRISQUE", "NIQE"])

display(dfDataset1Processed.head())

# Dataset 2
resultsDataset2Processed = []

for filename in dataset2Files:
    targetPath = dataset2Target + "/" + filename
    processedPath = outputFolder + "/dataset2/" + filename

    targetImage = cv2.imread(targetPath)
    processedImage = cv2.imread(processedPath)

    if targetImage is None or processedImage is None:
        continue

    PSNRValue = calculatePSNR(targetImage, processedImage)
    SSIMValue = calculateSSIM(targetImage, processedImage)
    BRISQUEValue = calculateBRISQUE(processedPath)
    NIQEValue = calculateNIQE(processedImage)

    resultsDataset2Processed.append(["dataset2", filename, "processed", PSNRValue, SSIMValue, BRISQUEValue, NIQEValue])

dfDataset2Processed = pd.DataFrame(resultsDataset2Processed, columns = ["dataset", "filename", "type", "PSNR", "SSIM", "BRISQUE", "NIQE"])

display(dfDataset2Processed.head())

,dataset,filename,type,PSNR,SSIM,BRISQUE,NIQE
0,dataset1,1.png,processed,27.870293,0.518614,NaN,112.769183
1,dataset1,111.png,processed,28.241116,0.405047,NaN,130.317752
2,dataset1,146.png,processed,27.975768,0.504909,NaN,122.571718
3,dataset1,179.png,processed,28.020684,0.780083,NaN,129.509830
4,dataset1,22.png,processed,28.275208,0.437185,NaN,125.959243


,dataset,filename,type,PSNR,SSIM,BRISQUE,NIQE
0,dataset2,uieb_10945.png,processed,27.639058,0.902865,NaN,59.876542
1,dataset2,uieb_112_img_.png,processed,27.678618,0.913339,NaN,43.803576
2,dataset2,uieb_117_img_.png,processed,28.041264,0.919902,NaN,22.401478
3,dataset2,uieb_120_img_.png,processed,27.990302,0.830882,NaN,36.666594
4,dataset2,uieb_12324.png,processed,27.815470,0.710098,NaN,74.571706


**Original vs processed**

In [ ]:
processedResults = pd.concat([dfDataset1Processed, dfDataset2Processed])

allResults = pd.concat([originalResults, processedResults])

summary = allResults.groupby(["dataset", "type"])[["PSNR", "SSIM", "BRISQUE", "NIQE"]].mean()

display(summary)

## Step 4 - Calculate Evaluation Metrics on Resized Data

### Part A

### Part B

Matthew's part